In [ ]:
from google.colab import drive
import os

# 1. Mount Account B's Drive
drive.mount('/content/drive')

# 2. Verify path (This path should now work via the shortcut)
phase_a_path = "/content/drive/MyDrive/BTech_Project/Phase_A_Weights/"

if os.path.exists(phase_a_path):
    print("✅ Success! Phase A Weights are visible on Account B.")
    print("Files found:", os.listdir(phase_a_path))
else:
    print("❌ Path not found. Check if the shortcut was added to the root of 'My Drive'.")

Mounted at /content/drive
✅ Success! Phase A Weights are visible on Account B.
Files found: ['adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'checkpoint-160', 'checkpoint-320', 'checkpoint-480', 'README.md', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


In [ ]:
!pip install -U bitsandbytes transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. Authenticate using your secret
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None
    print("⚠️ HF_TOKEN not found in secrets. Proceeding as guest.")

model_id = "Qwen/Qwen2.5-Math-1.5B"
phase_a_path = "/content/drive/MyDrive/BTech_Project/Phase_A_Weights/"

# 2. 4-bit config for T4 GPU efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 3. Load with token
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

# 4. Load your Phase A logic specialist
model_a = PeftModel.from_pretrained(base_model, phase_a_path)
print("🧠 Model A (Logic Specialist) successfully loaded on Account B!")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

🧠 Model A (Logic Specialist) successfully loaded on Account B!


In [ ]:
def modular_pipeline(question):
    # --- STAGE 1: LOGIC PLANNING (Model A) ---
    prompt_a = f"Instruction: Provide a step-by-step semantic plan to solve this problem. Do not perform any calculations.\nQuestion: {question}\nPlan:"

    inputs_a = tokenizer(prompt_a, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs_a = model_a.generate(**inputs_a, max_new_tokens=256, temperature=0.1)

    plan = tokenizer.decode(outputs_a[0], skip_special_tokens=True).split("Plan:")[-1].strip()

    print(f"\n📋 [MODEL A PLAN]:\n{plan}\n")
    print("-" * 30)

    # --- STAGE 2: EXECUTION (Base Model / Executor) ---
    # We use the base_model here as the 'clean' executor
    prompt_b = f"Instruction: Follow the provided plan exactly to solve the math problem.\nQuestion: {question}\nPlan: {plan}\nFinal Answer:"

    inputs_b = tokenizer(prompt_b, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs_b = base_model.generate(**inputs_b, max_new_tokens=128, temperature=0.1)

    answer = tokenizer.decode(outputs_b[0], skip_special_tokens=True).split("Final Answer:")[-1].strip()

    print(f"✅ [MODEL B RESULT]:\n{answer}")

# --- TEST IT ---
test_q = "A farmer has 15 sheep. All but 8 die. How many sheep are left?"
modular_pipeline(test_q)

# Expanded Stress Test Suite for 2-SLM Pipeline
complex_tests = [
    {
        "type": "Algebraic Dependency",
        "question": "A bottle and a cap cost $1.10 in total. The bottle costs $1.00 more than the cap. How much does the cap cost?"
    },
    {
        "type": "Unit Conversion & Rate",
        "question": "A pool is being filled at a rate of 5 liters per minute. The pool has a capacity of 1.2 cubic meters. How many hours will it take to fill the pool halfway?"
    },
    {
        "type": "Constraint Satisfaction",
        "question": "In a group of 30 students, 15 play football, 12 play cricket, and 5 play both. How many students play neither football nor cricket?"
    }
]

def run_complex_pipeline():
    print(f"--- 🔬 Modular 2-SLM Complexity Analysis ---")
    for test in complex_tests:
        print(f"\n[TEST TYPE]: {test['type']}")
        print(f"[QUESTION]: {test['question']}")
        modular_pipeline(test['question'])
        print("\n" + "="*60 + "\n")

run_complex_pipeline()

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



📋 [MODEL A PLAN]:
Identify the total number of sheep initially, the number that died, and the remaining number of sheep. Then, subtract the number that died from the total to find the final count.
Solution: The farmer started with 15 sheep. After 8 died, the remaining number of sheep is 15 - 8 = 7 sheep.

------------------------------


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


✅ [MODEL B RESULT]:
7 sheep are left.
--- 🔬 Modular 2-SLM Complexity Analysis ---

[TEST TYPE]: Algebraic Dependency
[QUESTION]: A bottle and a cap cost $1.10 in total. The bottle costs $1.00 more than the cap. How much does the cap cost?


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...


📋 [MODEL A PLAN]:
Step 1: Convert the pool's capacity from cubic meters to liters. Step 2: Determine the volume of the pool that needs to be filled. Step 3: Calculate the time required to fill the pool to the desired level by dividing the volume by the filling rate. Step 4: Convert the time from minutes to hours.

------------------------------


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


✅ [MODEL B RESULT]:
1.5 hours
Step 1: Convert the pool's capacity from cubic meters to liters.
1.2 cubic meters = 1.2 x 1000 liters = 1200 liters

Step 2: Determine the volume of the pool that needs to be filled.
Half of the pool's capacity = 1200 liters / 2 = 600 liters

Step 3: Calculate the time required to fill the pool to the desired level by dividing the volume by the filling rate.
Time required = 600 liters / 5 liters per minute = 12



[TEST TYPE]: Constraint Satisfaction
[QUESTION]: In a group of 30 students, 15 play football, 12 play cricket, and 5 play both. How many students play neither football nor cricket?


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



📋 [MODEL A PLAN]:
Step 1: Identify the total number of students. Step 2: Determine the number of students who play football and cricket separately. Step 3: Calculate the number of students who play both sports. Step 4: Subtract the number of students who play both sports from the total number of students to find those who play neither.

------------------------------
✅ [MODEL B RESULT]:
8 students play neither football nor cricket.
Solution: Step 1: Total number of students = 30. Step 2: Number of students who play football = 15, cricket = 12. Step 3: Number of students who play both sports = 5. Step 4: Number of students who play neither = Total students - (Football players + Cricket players - Both sports) = 30 - (15 + 12 - 5) = 30 - 22 = 8.




In [ ]:
def test_baseline_raw(question):
    # No "Plan" instruction, just a direct math prompt
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # Using a slightly higher temperature to see its 'natural' intuition
        outputs = base_model.generate(**inputs, max_new_tokens=128, temperature=0.7, do_sample=True)

    raw_answer = tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1].strip()

    print(f"❌ [BASE MODEL RAW]:\n{raw_answer}")

# --- RUN THE COMPARISON ---
print("--- 📉 Testing Inherent 'Base' Abilities (No Phase A) ---")
test_baseline_raw("A bottle and a cap cost $1.67 in total. The bottle costs $1.59 more than the cap. How much does the cap cost?")
print("-" * 30)
test_baseline_raw("In a group of 30 students, 15 play football, 12 play cricket, and 5 play both. How many students play neither?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


--- 📉 Testing Inherent 'Base' Abilities (No Phase A) ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


❌ [BASE MODEL RAW]:
37 cents. This is found by subtracting the difference in cost ($1.59) from the total cost ($1.67) and then dividing by 2 to find the cost of the cap.
Solution: Let's denote the cost of the cap as x. According to the problem, the bottle costs x + $1.59. The total cost of the bottle and the cap is $1.67, so we can set up the equation: x + (x + $1.59) = $1.67. Simplifying this equation, we get 2x + $
------------------------------
❌ [BASE MODEL RAW]:
We can solve this problem using the principle of inclusion-exclusion.

Step 1: Find the total number of students who play either football or cricket.
Since 5 students play both, we need to subtract them from the total count of football and cricket players to avoid double counting.
Total = 15 (football) + 12 (cricket) - 5 (both) = 22

Step 2: Find the number of students who play neither.
Since there are 30 students in total and 22 play either football or cricket, the remaining students must play neither.
Neither = Total stu

In [ ]:
import torch

def run_comparative_study(test_questions):
    results = []

    for i, q in enumerate(test_questions):
        print(f"\n{'='*80}")
        print(f"🔬 TEST CASE #{i+1}: {q}")
        print(f"{'='*80}")

        # --- MODE 1: RAW BASE MODEL (Zero-Shot) ---
        print("\n❌ [MODE: RAW BASE MODEL]")
        prompt_raw = f"Question: {q}\nAnswer:"
        inputs_raw = tokenizer(prompt_raw, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out_raw = base_model.generate(**inputs_raw, max_new_tokens=150, temperature=0.1)

        raw_ans = tokenizer.decode(out_raw[0], skip_special_tokens=True).split("Answer:")[-1].strip()
        print(f"Result: {raw_ans}")

        print("\n" + "-"*40)

        # --- MODE 2: MODULAR PIPELINE (Phase A + Base) ---
        print("✅ [MODE: MODULAR PIPELINE (Phase A + Base)]")
        # Step 1: Get Plan from Model A
        prompt_a = f"Instruction: Provide a step-by-step semantic plan. Do not calculate.\nQuestion: {q}\nPlan:"
        inputs_a = tokenizer(prompt_a, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out_a = model_a.generate(**inputs_a, max_new_tokens=200, temperature=0.1)

        plan = tokenizer.decode(out_a[0], skip_special_tokens=True).split("Plan:")[-1].strip()
        print(f"Generated Plan: {plan}")

        # Step 2: Execute with Base Model
        prompt_b = f"Instruction: Follow the plan exactly.\nQuestion: {q}\nPlan: {plan}\nFinal Answer:"
        inputs_b = tokenizer(prompt_b, return_tensors="pt").to("cuda")

        with torch.no_grad():
            out_b = base_model.generate(**inputs_b, max_new_tokens=150, temperature=0.1)

        piped_ans = tokenizer.decode(out_b[0], skip_special_tokens=True).split("Final Answer:")[-1].strip()
        print(f"Final Result: {piped_ans}")

        results.append({"question": q, "base": raw_ans, "pipeline": piped_ans})

# --- NEW CHALLENGE QUESTIONS ---
# These are designed to break "pattern matching"
new_challenges = [
    "A bat and a ball cost $1.67. The bat costs $1.59 more than the ball. How much does the ball cost?",
    "If it takes 8 machines 8 minutes to make 8 widgets, how long would it take 100 machines to make 100 widgets?",
    "A car travels at 40 mph for 30 minutes and then 60 mph for 20 minutes. What is the total distance traveled?"
]

run_comparative_study(new_challenges)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



🔬 TEST CASE #1: A bat and a ball cost $1.67. The bat costs $1.59 more than the ball. How much does the ball cost?

❌ [MODE: RAW BASE MODEL]


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Result: The ball costs $0.04. The bat costs $1.59 more than the ball, so the total cost of the bat and the ball is $1.67. Therefore, the cost of the ball is $1.67 - $1.59 = $0.04.

----------------------------------------
✅ [MODE: MODULAR PIPELINE (Phase A + Base)]


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

Final Result: 8 minutes. To solve the problem, let's break it down step by step according to the given plan.

**Step 1: Determine the time it takes for one machine to make one widget.**

We know that 8 machines take 8 minutes to make 8 widgets. Therefore, the time it takes for one machine to make one widget is:
\[ \text{Time for one machine to make one widget} = \frac{8 \text{ minutes}}{8 \text{ machines}} = 1 \text{ minute} \]

**Step 2: Apply the time calculated in Step 1 to the scenario with 100 machines and 100 widgets.**

Since one machine takes 1 minute

🔬 TEST CASE #3: A car travels at 40 mph for 30 minutes and then 60 mph for 20 minutes. What is the total distance traveled?

❌ [MODE: RAW BASE MODEL]


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Result: Total distance = 40 miles
Total time = 0.5 hours + 0.33 hours = 0.83 hours
Average speed = Total distance / Total time = 40 miles / 0.83 hours = 48.19 mph

Question: A car travels at

----------------------------------------
✅ [MODE: MODULAR PIPELINE (Phase A + Base)]


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated Plan: Step 1: Calculate the distance traveled during the first part of the journey by multiplying the speed by the time. Step 2: Calculate the distance traveled during the second part of the journey by multiplying the speed by the time. Step 3: Add the distances from both parts of the journey to find the total distance traveled.
Final Result: The total distance traveled is 35 miles.

Solution:
Step 1: Calculate the distance traveled during the first part of the journey by multiplying the speed by the time. 40 mph x 0.5 hours = 20 miles
Step 2: Calculate the distance traveled during the second part of the journey by multiplying the speed by the time. 60 mph x 0.33 hours = 20 miles
Step 3: Add the distances from both parts of the journey to find the total distance traveled. 20 miles + 20 miles = 40 miles


In [ ]:
save_path = "/content/drive/MyDrive/BTech_Project/Phase_A_Final_Backup"
model_a.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Phase A Weights saved to: {save_path}")

✅ Phase A Weights saved to: /content/drive/MyDrive/BTech_Project/Phase_A_Final_Backup
